In [ ]:
# Stop warnings
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import numpy as np
import pandas as pd

# Figure imports
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Personal imports
sys.path.append("{}/../../../../analysis_code/utils".format(os.getcwd()))
from settings_utils import load_settings
from plot_utils import *
from maths_utils import weighted_nan_median, weighted_nan_percentile
from surface_utils import load_surface, make_surface_image
from pycortex_utils import set_pycortex_config_file, get_rois


In [ ]:
# Inputs
main_dir = '/Users/uriel/disks/meso_shared'
project_dir = 'nCSF'
subject = 'sub-02'
format_ = 'fsnative'
avg_method = 'loo-avg'
ncsf_task_name = 'nCSF'
rois_method_format = 'rois-group-mmp'
rsq2use = 'ncsf_loo_rsq'

In [ ]:
# Set pycortex db and colormaps
cortex_dir = "{}/{}/derivatives/pp_data/cortex".format(main_dir, project_dir)
set_pycortex_config_file(cortex_dir)

In [ ]:
# Load settings
base_dir = os.path.abspath(os.path.join(os.getcwd(), "../../../../"))
settings_path = os.path.join(base_dir, project_dir, "settings.yml")
prf_settings_path = os.path.join(base_dir, project_dir, "prf-analysis.yml")
ncsf_settings_path = os.path.join(base_dir, project_dir, "nCSF-analysis.yml")
figure_settings_path = os.path.join(base_dir, project_dir, "figure-settings.yml")
settings = load_settings([settings_path, prf_settings_path, figure_settings_path, ncsf_settings_path])
analysis_info = settings[0]
max_SFp = analysis_info['flatmap_ncsf_SFp_scale'][1]
max_auc = analysis_info['flatmap_ncsf_auc_scale'][1]

ecc_size_max = analysis_info['ecc_size_max']
stats_threshold = analysis_info['stats_th']
ecc_size_num_bins = 16

In [ ]:
if rois_method_format == 'rois-drawn':
    rois = analysis_info[rois_method_format]
elif rois_method_format == 'rois-group-mmp':
    rois = list(analysis_info[rois_method_format].keys())

if rois_method_format == 'rois-drawn':
    analysis_info['rois'] = analysis_info[rois_method_format]
elif rois_method_format == 'rois-group-mmp':
    analysis_info['rois'] = list(analysis_info[rois_method_format].keys())

In [ ]:
ncsf_tsv_dir = '{}/{}/derivatives/pp_data/{}/{}/ncsf/tsv'.format(main_dir, project_dir, subject, format_)
ncsf_tsv_fn = '{}/{}_task-{}_fmriprep_dct_z-score_{}_{}_ncsf_deriv.tsv'.format(ncsf_tsv_dir, subject, ncsf_task_name, avg_method, rois_method_format)

In [ ]:
ncsf_tsv_df = pd.read_table(ncsf_tsv_fn, sep="\t")

In [ ]:
if stats_threshold == 0.05: stats_col = 'nCSF_corr_pvalue_5pt'
elif stats_threshold == 0.01: stats_col = 'nCSF_corr_pvalue_1pt'
ncsf_tsv_df.loc[
        #  (data.amplitude < amplitude_threshold[0]) |
        #  (data.prf_ecc < ecc_threshold[0]) | (data.prf_ecc > ecc_threshold[1]) |
         #(data.prf_size < size_threshold[0]) | 
    # (ncsf_tsv_df.prf_size > 5) |
        #  (data.prf_n < n_threshold[0]) | (data.prf_n > n_threshold[1]) | 
        #  (data[rsq2use] < rsqr_threshold) |
         (ncsf_tsv_df[stats_col] > stats_threshold)] = np.nan

ncsf_tsv_df = ncsf_tsv_df.dropna()

In [ ]:
ncsf_tsv_df.columns

In [ ]:
# median rsq between nCSf and pRF
ncsf_tsv_df['ncsf_prf_median_loo_rsq'] = (ncsf_tsv_df[['ncsf_loo_rsq', 'prf_loo_rsq']].median(axis=1))

# ECC SFp

In [ ]:
# ecc_bins = np.linspace(0, ecc_size_max[0], ecc_size_num_bins+1) 

x = np.linspace(0, 1, ecc_size_num_bins + 1)
ecc_bins = (x**2) * ecc_size_max[0]


for num_roi, roi in enumerate(rois):
    df_roi = ncsf_tsv_df.loc[(ncsf_tsv_df.roi == roi)]
    df_bins = df_roi.groupby(pd.cut(df_roi['prf_ecc'], bins=ecc_bins, include_lowest=True), observed=False)
    df_ecc_SFp_bin = pd.DataFrame()
    df_ecc_SFp_bin['roi'] = [roi]*ecc_size_num_bins
    df_ecc_SFp_bin['num_bins'] = np.arange(ecc_size_num_bins)  
    df_ecc_SFp_bin['prf_ecc_bins'] = df_bins.apply(lambda x: weighted_nan_median(x['prf_ecc'].values, x[rsq2use].values)).values
    df_ecc_SFp_bin['SFp_bins_median'] = df_bins.apply(lambda x: weighted_nan_median(x['SFp'].values, x[rsq2use].values)).values
    df_ecc_SFp_bin[f'{rsq2use}_bins_median'] = np.array(df_bins[rsq2use].median())
    df_ecc_SFp_bin['SFp_bins_ci_upper_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['SFp'].values, x[rsq2use].values, 75)).values
    df_ecc_SFp_bin['SFp_bins_ci_lower_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['SFp'].values, x[rsq2use].values, 25)).values
    if num_roi == 0: df_ecc_SFp_bins = df_ecc_SFp_bin
    else: df_ecc_SFp_bins = pd.concat([df_ecc_SFp_bins, df_ecc_SFp_bin]) 

df_ecc_SFp = df_ecc_SFp_bins

In [ ]:
result = df_bins.apply(lambda x: weighted_nan_median(x['prf_ecc'].values, x[rsq2use].values))
print(roi, len(result))

In [ ]:
print(df_roi['prf_ecc'].max())  # > 12 ?

In [ ]:
def prf_ecc_size_plot(df, figure_info, rsq2use):
    """
    Make scatter plot for linear relationship between eccentricity and size

    Parameters
    ----------
    df : A data dataframe
    figure_info : dict with figure settings
    rsq2use : rsquare to use
    
    Returns
    -------
    fig : eccentricy as a function of size plot
    """
    
    from maths_utils import weighted_regression

    # General figure settings
    template_specs = dict(axes_color="rgba(0, 0, 0, 1)",
                          axes_width=2,
                          axes_font_size=15,
                          bg_col="rgba(255, 255, 255, 1)",
                          font='Arial',
                          title_font_size=15,
                          rois_plot_width=1.5)
    
    # General figure settings
    fig_template = plotly_template(template_specs)
    max_size = figure_info['ecc_size_max'][1]
    roi_colors = figure_info['roi_colors']
    fig_margin = figure_info['rois_fig_margin']
    rois_groups = figure_info['rois_groups_plot']
    rows, cols = 1, len(rois_groups)
    rois_hor_spacing = figure_info['rois_hor_spacing']
    rois_ver_spacing = figure_info['rois_ver_spacing']
    rois_plot_height = figure_info['rois_plot_height']
    rois_plot_width = figure_info['rois_plot_width']    
    ecc_axis = [0, figure_info['ecc_size_max'][0]]
    size_axis = [0, analysis_info['flatmap_ncsf_SFp_scale'][1]]
    
    fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows-1))
    fig_width = rois_plot_width * cols + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols-1))
    hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
    ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])

    # General settings
    
    fig = make_subplots(rows=rows, cols=cols, print_grid=False, 
                       horizontal_spacing=hor_spacing,
                       vertical_spacing=ver_spacing)
    
    for l, line_label in enumerate(rois_groups):
        for j, roi in enumerate(line_label):
            
            # Parametring colors
            roi_color = roi_colors[roi]
            roi_color_opac = f"rgba{roi_color[3:-1]}, 0.15)"
            
            # Get data
            df_roi = df.loc[(df.roi == roi)]
            ecc_median = np.array(df_roi.prf_ecc_bins)
            SFp_median = np.array(df_roi.SFp_bins_median)
            r2_median = np.array(df_roi[f'{rsq2use}_bins_median'])
            SFp_upper_bound = np.array(df_roi.SFp_bins_ci_upper_bound)
            SFp_lower_bound = np.array(df_roi.SFp_bins_ci_lower_bound)
            
            # # Linear regression
            # slope, intercept = weighted_regression(ecc_median, SFp_median, r2_median, model='linear')
            # slope_upper, intercept_upper = weighted_regression(ecc_median[np.where(~np.isnan(SFp_upper_bound))], 
            #                                                    SFp_upper_bound[~np.isnan(SFp_upper_bound)], 
            #                                                    r2_median[np.where(~np.isnan(SFp_upper_bound))], 
            #                                                    model='linear')
            # slope_lower, intercept_lower = weighted_regression(ecc_median[np.where(~np.isnan(SFp_lower_bound))], 
            #                                                    SFp_lower_bound[~np.isnan(SFp_lower_bound)], 
            #                                                    r2_median[np.where(~np.isnan(SFp_lower_bound))], 
            #                                                    model='linear')

            # line_x = np.linspace(ecc_median[0], ecc_median[-1], 50)
            # line = slope * line_x + intercept
            # line_upper = slope_upper * line_x + intercept_upper
            # line_lower = slope_lower * line_x + intercept_lower

            # fig.add_trace(go.Scatter(x=line_x, y=line, mode='lines', name=roi, legendgroup=roi, 
            #                           line=dict(color=roi_color, width=3), showlegend=False), 
            #               row=1, col=l+1)

            # # Error area
            # fig.add_trace(go.Scatter(x=np.concatenate([line_x, line_x[::-1]]), 
            #                           y=np.concatenate([list(line_upper), list(line_lower[::-1])]), 
            #                           mode='lines', fill='toself', fillcolor=roi_color_opac, 
            #                           line=dict(color=roi_color_opac, width=0), showlegend=False), 
            #               row=1, col=l+1)

            # Markers
            fig.add_trace(go.Scatter(x=ecc_median, 
                                     y=SFp_median, 
                                     mode='markers', 
                                     error_y=dict(type='data', 
                                                  array=SFp_upper_bound - SFp_median, 
                                                  arrayminus=SFp_median - SFp_lower_bound,
                                                  visible=True, 
                                                  thickness=3, 
                                                  width=0, 
                                                  color=roi_color),
                                      marker=dict(color=roi_color,
                                                  symbol='square',
                                                  size=8, 
                                                  line=dict(color=roi_color, 
                                                            width=3)), 
                                      showlegend=False), 
                          row=1, col=l + 1)
            
            # Add legend
            annotation = go.layout.Annotation(x=1, y=max_SFp-(j*0.1*max_SFp), text=roi, xanchor='left',
                                              showarrow=False, font_color=roi_color, 
                                              font_family=template_specs['font'],
                                              font_size=template_specs['axes_font_size'],
                                             )
            fig.add_annotation(annotation, row=1, col=l+1)

        # Set axis
        fig.update_xaxes(title_text='pRF eccentricity (dva)', range=[ecc_axis[0], ecc_axis[1]], showline=True)
        fig.update_yaxes(title_text='pRF SFp (dva)', range=[size_axis[0], size_axis[1]], showline=True)
        fig.update_layout(height=fig_height, 
                          width=fig_width, 
                          showlegend=False, 
                          template=fig_template,
                          margin_l=fig_margin[0], 
                          margin_t=fig_margin[1], 
                          margin_r=fig_margin[2], 
                          margin_b=fig_margin[3]
                         )
        
    return fig

In [ ]:
fig = prf_ecc_size_plot(df=df_ecc_SFp, figure_info=analysis_info, rsq2use=rsq2use)
fig.show()

# ECC auc

In [ ]:
x = np.linspace(0, 1, ecc_size_num_bins + 1)
ecc_bins = (x**2) * ecc_size_max[0]

for num_roi, roi in enumerate(rois):
    df_roi = ncsf_tsv_df.loc[(ncsf_tsv_df.roi == roi)]
    df_bins = df_roi.groupby(pd.cut(df_roi['prf_ecc'], bins=ecc_bins, include_lowest=True), observed=False)
    df_ecc_auc_bin = pd.DataFrame()
    df_ecc_auc_bin['roi'] = [roi]*ecc_size_num_bins
    df_ecc_auc_bin['num_bins'] = np.arange(ecc_size_num_bins)  
    df_ecc_auc_bin['prf_ecc_bins'] = df_bins.apply(lambda x: weighted_nan_median(x['prf_ecc'].values, x[rsq2use].values)).values
    df_ecc_auc_bin['auc_bins_median'] = df_bins.apply(lambda x: weighted_nan_median(x['auc'].values, x[rsq2use].values)).values
    df_ecc_auc_bin[f'{rsq2use}_bins_median'] = np.array(df_bins[rsq2use].median())
    df_ecc_auc_bin['auc_bins_ci_upper_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['auc'].values, x[rsq2use].values, 75)).values
    df_ecc_auc_bin['auc_bins_ci_lower_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['auc'].values, x[rsq2use].values, 25)).values
    if num_roi == 0: df_ecc_auc_bins = df_ecc_auc_bin
    else: df_ecc_auc_bins = pd.concat([df_ecc_auc_bins, df_ecc_auc_bin]) 

df_ecc_auc = df_ecc_auc_bins

In [ ]:
def prf_ecc_auc_plot(df, figure_info, rsq2use):
    """
    Make scatter plot for linear relationship between eccentricity and size

    Parameters
    ----------
    df : A data dataframe
    figure_info : dict with figure settings
    rsq2use : rsquare to use
    
    Returns
    -------
    fig : eccentricy as a function of size plot
    """
    
    from maths_utils import weighted_regression

    # General figure settings
    template_specs = dict(axes_color="rgba(0, 0, 0, 1)",
                          axes_width=2,
                          axes_font_size=15,
                          bg_col="rgba(255, 255, 255, 1)",
                          font='Arial',
                          title_font_size=15,
                          rois_plot_width=1.5)
    
    # General figure settings
    fig_template = plotly_template(template_specs)
    max_size = figure_info['ecc_size_max'][1]
    roi_colors = figure_info['roi_colors']
    fig_margin = figure_info['rois_fig_margin']
    rois_groups = figure_info['rois_groups_plot']
    rows, cols = 1, len(rois_groups)
    rois_hor_spacing = figure_info['rois_hor_spacing']
    rois_ver_spacing = figure_info['rois_ver_spacing']
    rois_plot_height = figure_info['rois_plot_height']
    rois_plot_width = figure_info['rois_plot_width']    
    ecc_axis = [0, figure_info['ecc_size_max'][0]]
    size_axis = [0, analysis_info['flatmap_ncsf_auc_scale'][1]]
    
    fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows-1))
    fig_width = rois_plot_width * cols + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols-1))
    hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
    ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])

    # General settings
    
    fig = make_subplots(rows=rows, cols=cols, print_grid=False, 
                       horizontal_spacing=hor_spacing,
                       vertical_spacing=ver_spacing)
    
    for l, line_label in enumerate(rois_groups):
        for j, roi in enumerate(line_label):
            
            # Parametring colors
            roi_color = roi_colors[roi]
            roi_color_opac = f"rgba{roi_color[3:-1]}, 0.15)"
            
            # Get data
            df_roi = df.loc[(df.roi == roi)]
            ecc_median = np.array(df_roi.prf_ecc_bins)
            auc_median = np.array(df_roi.auc_bins_median)
            r2_median = np.array(df_roi[f'{rsq2use}_bins_median'])
            auc_upper_bound = np.array(df_roi.auc_bins_ci_upper_bound)
            auc_lower_bound = np.array(df_roi.auc_bins_ci_lower_bound)
            
            # Markers
            fig.add_trace(go.Scatter(x=ecc_median, 
                                     y=auc_median, 
                                     mode='markers', 
                                     error_y=dict(type='data', 
                                                  array=auc_upper_bound - auc_median, 
                                                  arrayminus=auc_median - auc_lower_bound,
                                                  visible=True, 
                                                  thickness=3, 
                                                  width=0, 
                                                  color=roi_color),
                                      marker=dict(color=roi_color,
                                                  symbol='square',
                                                  size=8, 
                                                  line=dict(color=roi_color, 
                                                            width=3)), 
                                      showlegend=False), 
                          row=1, col=l + 1)
            
            # Add legend
            annotation = go.layout.Annotation(x=1, y=max_auc - (j*0.1*max_auc), text=roi, xanchor='left',
                                              showarrow=False, font_color=roi_color, 
                                              font_family=template_specs['font'],
                                              font_size=template_specs['axes_font_size'],
                                             )
            fig.add_annotation(annotation, row=1, col=l+1)

        # Set axis
        fig.update_xaxes(title_text='pRF eccentricity (dva)', range=[ecc_axis[0], ecc_axis[1]], showline=True)
        fig.update_yaxes(title_text='nCSF auc', range=[size_axis[0],10], showline=True)
        fig.update_layout(height=fig_height, 
                          width=fig_width, 
                          showlegend=False, 
                          template=fig_template,
                          margin_l=fig_margin[0], 
                          margin_t=fig_margin[1], 
                          margin_r=fig_margin[2], 
                          margin_b=fig_margin[3]
                         )
        
    return fig

In [ ]:
fig = prf_ecc_auc_plot(df=df_ecc_auc, figure_info=analysis_info, rsq2use=rsq2use)
fig.show()

# Size SFp

In [ ]:
def size_sfp_plot(df, figure_info, rsq2use):
    """
    Make scatter plot for linear relationship between eccentricity and size

    Parameters
    ----------
    df : A data dataframe
    figure_info : dict with figure settings
    rsq2use : rsquare to use
    
    Returns
    -------
    fig : eccentricy as a function of size plot
    """
    
    from maths_utils import weighted_regression

    # General figure settings
    template_specs = dict(axes_color="rgba(0, 0, 0, 1)",
                          axes_width=2,
                          axes_font_size=15,
                          bg_col="rgba(255, 255, 255, 1)",
                          font='Arial',
                          title_font_size=15,
                          rois_plot_width=1.5)
    
    # General figure settings
    fig_template = plotly_template(template_specs)
    max_size = figure_info['ecc_size_max'][1]
    roi_colors = figure_info['roi_colors']
    fig_margin = figure_info['rois_fig_margin']
    rois_groups = figure_info['rois_groups_plot']
    rows, cols = 1, len(rois_groups)
    rois_hor_spacing = figure_info['rois_hor_spacing']
    rois_ver_spacing = figure_info['rois_ver_spacing']
    rois_plot_height = figure_info['rois_plot_height']
    rois_plot_width = figure_info['rois_plot_width']    
    size_axis = [0, figure_info['ecc_size_max'][0]]
    # size_axis = [0, analysis_info['flatmap_ncsf_SFp_scale'][1]]
    
    fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows-1))
    fig_width = rois_plot_width * cols + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols-1))
    hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
    ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])

    # General settings
    
    fig = make_subplots(rows=rows, cols=cols, print_grid=False, 
                       horizontal_spacing=hor_spacing,
                       vertical_spacing=ver_spacing)
    
    for l, line_label in enumerate(rois_groups):
        for j, roi in enumerate(line_label):
            
            # Parametring colors
            roi_color = roi_colors[roi]
            roi_color_opac = f"rgba{roi_color[3:-1]}, 0.15)"
            
            # Get data
            df_roi = df.loc[(df.roi == roi)]
            size_median = np.array(df_roi.prf_size_bins)
            SFp_median = np.array(df_roi.sfp_bins_median)
            r2_median = np.array(df_roi[f'{rsq2use}_bins_median'])
            SFp_upper_bound = np.array(df_roi.sfp_bins_ci_upper_bound)
            SFp_lower_bound = np.array(df_roi.sfp_bins_ci_lower_bound)
            # Markers
            fig.add_trace(go.Scatter(x=size_median, 
                                     y=SFp_median, 
                                     mode='markers', 
                                     error_y=dict(type='data', 
                                                  array=SFp_upper_bound - SFp_median, 
                                                  arrayminus=SFp_median - SFp_lower_bound,
                                                  visible=True, 
                                                  thickness=3, 
                                                  width=0, 
                                                  color=roi_color),
                                      marker=dict(color=roi_color,
                                                  symbol='square',
                                                  size=8, 
                                                  line=dict(color=roi_color, 
                                                            width=3)), 
                                      showlegend=False), 
                          row=1, col=l + 1)
            
            # Add legend
            annotation = go.layout.Annotation(x=1, y=max_SFp-(j*0.1*max_SFp), text=roi, xanchor='left',
                                              showarrow=False, font_color=roi_color, 
                                              font_family=template_specs['font'],
                                              font_size=template_specs['axes_font_size'],
                                             )
            fig.add_annotation(annotation, row=1, col=l+1)

        # Set axis
        fig.update_xaxes(title_text='pRF size (dva)', range=[size_axis[0], size_axis[1]], showline=True)
        fig.update_yaxes(title_text='pRF SFp', range=[size_axis[0], size_axis[1]], showline=True)
        fig.update_layout(height=fig_height, 
                          width=fig_width, 
                          showlegend=False, 
                          template=fig_template,
                          margin_l=fig_margin[0], 
                          margin_t=fig_margin[1], 
                          margin_r=fig_margin[2], 
                          margin_b=fig_margin[3]
                         )
        
    return fig

In [ ]:
size_bins = np.linspace(0, ecc_size_max[0], ecc_size_num_bins + 1)

# x = np.linspace(0, 1, ecc_size_num_bins + 1)
# size_bins = (x**2) * ecc_size_max[0]

for num_roi, roi in enumerate(rois):
    df_roi = ncsf_tsv_df.loc[(ncsf_tsv_df.roi == roi)]
    df_bins = df_roi.groupby(pd.cut(df_roi['prf_size'], bins=size_bins, include_lowest=True), observed=False)
    df_size_sfp_bin = pd.DataFrame()
    df_size_sfp_bin['roi'] = [roi]*ecc_size_num_bins
    df_size_sfp_bin['num_bins'] = np.arange(ecc_size_num_bins)  
    df_size_sfp_bin['prf_size_bins'] = df_bins.apply(lambda x: weighted_nan_median(x['prf_size'].values, x[rsq2use].values)).values
    df_size_sfp_bin['sfp_bins_median'] = df_bins.apply(lambda x: weighted_nan_median(x['SFp'].values, x[rsq2use].values)).values
    df_size_sfp_bin[f'{rsq2use}_bins_median'] = np.array(df_bins[rsq2use].median())
    df_size_sfp_bin['sfp_bins_ci_upper_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['SFp'].values, x[rsq2use].values, 75)).values
    df_size_sfp_bin['sfp_bins_ci_lower_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['SFp'].values, x[rsq2use].values, 25)).values
    if num_roi == 0: df_size_sfp_bins = df_size_sfp_bin
    else: df_size_sfp_bins = pd.concat([df_size_sfp_bins, df_size_sfp_bin]) 

df_size_sfp = df_size_sfp_bins

In [ ]:
fig = size_sfp_plot(df=df_size_sfp, figure_info=analysis_info, rsq2use=rsq2use)
fig.show()

# size auc

In [ ]:
def size_auc_plot(df, figure_info, rsq2use):
    """
    Make scatter plot for linear relationship between eccentricity and size

    Parameters
    ----------
    df : A data dataframe
    figure_info : dict with figure settings
    rsq2use : rsquare to use
    
    Returns
    -------
    fig : eccentricy as a function of size plot
    """
    
    from maths_utils import weighted_regression

    # General figure settings
    template_specs = dict(axes_color="rgba(0, 0, 0, 1)",
                          axes_width=2,
                          axes_font_size=15,
                          bg_col="rgba(255, 255, 255, 1)",
                          font='Arial',
                          title_font_size=15,
                          rois_plot_width=1.5)
    
    # General figure settings
    fig_template = plotly_template(template_specs)
    max_size = figure_info['ecc_size_max'][1]
    roi_colors = figure_info['roi_colors']
    fig_margin = figure_info['rois_fig_margin']
    rois_groups = figure_info['rois_groups_plot']
    rows, cols = 1, len(rois_groups)
    rois_hor_spacing = figure_info['rois_hor_spacing']
    rois_ver_spacing = figure_info['rois_ver_spacing']
    rois_plot_height = figure_info['rois_plot_height']
    rois_plot_width = figure_info['rois_plot_width']    
    size_axis = [0, figure_info['ecc_size_max'][0]]
    # size_axis = [0, analysis_info['flatmap_ncsf_SFp_scale'][1]]
    
    fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows-1))
    fig_width = rois_plot_width * cols + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols-1))
    hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
    ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])

    # General settings
    
    fig = make_subplots(rows=rows, cols=cols, print_grid=False, 
                       horizontal_spacing=hor_spacing,
                       vertical_spacing=ver_spacing)
    
    for l, line_label in enumerate(rois_groups):
        for j, roi in enumerate(line_label):
            
            # Parametring colors
            roi_color = roi_colors[roi]
            roi_color_opac = f"rgba{roi_color[3:-1]}, 0.15)"
            
            # Get data
            df_roi = df.loc[(df.roi == roi)]
            size_median = np.array(df_roi.prf_size_bins)
            auc_median = np.array(df_roi.auc_bins_median)
            r2_median = np.array(df_roi[f'{rsq2use}_bins_median'])
            auc_upper_bound = np.array(df_roi.auc_bins_ci_upper_bound)
            auc_lower_bound = np.array(df_roi.auc_bins_ci_lower_bound)
            # Markers
            fig.add_trace(go.Scatter(x=size_median, 
                                     y=auc_median, 
                                     mode='markers', 
                                     error_y=dict(type='data', 
                                                  array=auc_upper_bound - auc_median, 
                                                  arrayminus=auc_median - auc_lower_bound,
                                                  visible=True, 
                                                  thickness=3, 
                                                  width=0, 
                                                  color=roi_color),
                                      marker=dict(color=roi_color,
                                                  symbol='square',
                                                  size=8, 
                                                  line=dict(color=roi_color, 
                                                            width=3)), 
                                      showlegend=False), 
                          row=1, col=l + 1)
            
            # Add legend
            annotation = go.layout.Annotation(x=1, y=max_auc-(j*0.1*max_auc), text=roi, xanchor='left',
                                              showarrow=False, font_color=roi_color, 
                                              font_family=template_specs['font'],
                                              font_size=template_specs['axes_font_size'],
                                             )
            fig.add_annotation(annotation, row=1, col=l+1)

        # Set axis
        fig.update_xaxes(title_text='pRF size (dva)', range=[size_axis[0], size_axis[1]], showline=True)
        fig.update_yaxes(title_text='pRF auc', range=[size_axis[0], size_axis[1]], showline=True)
        fig.update_layout(height=fig_height, 
                          width=fig_width, 
                          showlegend=False, 
                          template=fig_template,
                          margin_l=fig_margin[0], 
                          margin_t=fig_margin[1], 
                          margin_r=fig_margin[2], 
                          margin_b=fig_margin[3]
                         )
        
    return fig

In [ ]:
size_bins = np.linspace(0, ecc_size_max[0], ecc_size_num_bins + 1)

for num_roi, roi in enumerate(rois):
    df_roi = ncsf_tsv_df.loc[(ncsf_tsv_df.roi == roi)]
    df_bins = df_roi.groupby(pd.cut(df_roi['prf_size'], bins=size_bins, include_lowest=True), observed=False)
    df_size_auc_bin = pd.DataFrame()
    df_size_auc_bin['roi'] = [roi]*ecc_size_num_bins
    df_size_auc_bin['num_bins'] = np.arange(ecc_size_num_bins)  
    df_size_auc_bin['prf_size_bins'] = df_bins.apply(lambda x: weighted_nan_median(x['prf_size'].values, x[rsq2use].values)).values
    df_size_auc_bin['auc_bins_median'] = df_bins.apply(lambda x: weighted_nan_median(x['auc'].values, x[rsq2use].values)).values
    df_size_auc_bin[f'{rsq2use}_bins_median'] = np.array(df_bins[rsq2use].median())
    df_size_auc_bin['auc_bins_ci_upper_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['auc'].values, x[rsq2use].values, 75)).values
    df_size_auc_bin['auc_bins_ci_lower_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['auc'].values, x[rsq2use].values, 25)).values
    if num_roi == 0: df_size_auc_bins = df_size_auc_bin
    else: df_size_auc_bins = pd.concat([df_size_auc_bins, df_size_auc_bin]) 

df_size_auc = df_size_auc_bins

In [ ]:
fig = size_auc_plot(df=df_size_auc, figure_info=analysis_info, rsq2use=rsq2use)
fig.show()

# polar pCM

In [ ]:
def prf_polar_pcm_plot(df, figure_info, rsq2use):
    """
    Make scatter plot for linear relationship between eccentricity and size

    Parameters
    ----------
    df : A data dataframe
    figure_info : dict with figure settings
    rsq2use : rsquare to use
    
    Returns
    -------
    fig : eccentricy as a function of size plot
    """
    
    from maths_utils import weighted_regression

    max_ecc = 10 #############
    # General figure settings
    template_specs = dict(axes_color="rgba(0, 0, 0, 1)",
                          axes_width=2,
                          axes_font_size=15,
                          bg_col="rgba(255, 255, 255, 1)",
                          font='Arial',
                          title_font_size=15,
                          rois_plot_width=1.5)
    
    # General figure settings
    fig_template = plotly_template(template_specs)
    max_size = figure_info['ecc_size_max'][1]
    roi_colors = figure_info['roi_colors']
    fig_margin = figure_info['rois_fig_margin']
    rois_groups = figure_info['rois_groups_plot']
    rows, cols = 1, len(rois_groups)
    rois_hor_spacing = figure_info['rois_hor_spacing']
    rois_ver_spacing = figure_info['rois_ver_spacing']
    rois_plot_height = figure_info['rois_plot_height']
    rois_plot_width = figure_info['rois_plot_width']    
    ecc_axis = [0, figure_info['ecc_size_max'][0]]
    size_axis = [0, analysis_info['flatmap_ncsf_auc_scale'][1]]
    
    fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows-1))
    fig_width = rois_plot_width * cols + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols-1))
    hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
    ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])

    # General settings
    
    fig = make_subplots(rows=rows, cols=cols, print_grid=False, 
                       horizontal_spacing=hor_spacing,
                       vertical_spacing=ver_spacing)
    
    for l, line_label in enumerate(rois_groups):
        for j, roi in enumerate(line_label):
            
            # Parametring colors
            roi_color = roi_colors[roi]
            roi_color_opac = f"rgba{roi_color[3:-1]}, 0.15)"
            
            # Get data
            df_roi = df.loc[(df.roi == roi)]
            polar_median = np.array(df_roi.prf_polar_bins)
            cm_median = np.array(df_roi.prf_cm_bins_median)
            r2_median = np.array(df_roi[f'{rsq2use}_bins_median'])
            cm_upper_bound = np.array(df_roi.prf_cm_bins_ci_upper_bound)
            cm_lower_bound = np.array(df_roi.prf_cm_bins_ci_lower_bound)
            

            # Markers
            fig.add_trace(go.Scatter(x=polar_median, 
                                     y=cm_median, 
                                     mode='markers+lines', 
                                     error_y=dict(type='data', 
                                                  array=cm_upper_bound - cm_median, 
                                                  arrayminus=cm_median - cm_lower_bound,
                                                  visible=True, 
                                                  thickness=3, 
                                                  width=0, 
                                                  color=roi_color),
                                      marker=dict(color=roi_color,
                                                  symbol='square',
                                                  size=8, 
                                                  line=dict(color=roi_color, 
                                                            width=3)), 
                                      showlegend=False), 
                          row=1, col=l + 1)
            
            # Add legend
            annotation = go.layout.Annotation(x=1, y=max_ecc-j*1.5, text=roi, xanchor='left',
                                              showarrow=False, font_color=roi_color, 
                                              font_family=template_specs['font'],
                                              font_size=template_specs['axes_font_size'],
                                             )
            fig.add_annotation(annotation, row=1, col=l+1)

        # Set axis titles only for the left-most column and bottom-most row
        fig.update_yaxes(title_text='pRF cortical magn. (mm/dva)', row=1, col=1)
        fig.update_xaxes(title_text='pRF polar angle (dva)', range=[0, 360], tickvals=[0, 90, 180, 270, 360], ticktext=['0', '90', '180', '270', '360'], showline=True, row=1, col=l+1)
        fig.update_yaxes(range=[0, 10], showline=True)
        fig.update_layout(height=fig_height, 
                          width=fig_width, 
                          showlegend=False, 
                          template=fig_template,
                          margin_l=fig_margin[0], 
                          margin_t=fig_margin[1], 
                          margin_r=fig_margin[2], 
                          margin_b=fig_margin[3]
                         )
        
    return fig

In [ ]:
data = ncsf_tsv_df
num_polar_cm_bins = 8
data['prf_polar_angle'] = np.mod(np.degrees(np.angle(data.polar_real + 1j * data.polar_imag)), 360)
for num_roi, roi in enumerate(rois):
    df_roi = data.loc[(data.roi == roi)]
    df_bins = df_roi.groupby(pd.cut(df_roi['prf_polar_angle'], bins=num_polar_cm_bins))
    df_polar_cm_bin = pd.DataFrame()
    df_polar_cm_bin['roi'] = [roi]*num_polar_cm_bins
    df_polar_cm_bin['num_bins'] = np.arange(num_polar_cm_bins)
    df_polar_cm_bin['prf_polar_bins'] = df_bins.apply(lambda x: weighted_nan_median(x['prf_polar_angle'].values, x['prf_loo_rsq'].values)).values
    df_polar_cm_bin['prf_cm_bins_median'] = df_bins.apply(lambda x: weighted_nan_median(x['pcm_median'].values, x['prf_loo_rsq'].values)).values
    df_polar_cm_bin['prf_loo_rsq_bins_median'] = np.array(df_bins['prf_loo_rsq'].median())
    df_polar_cm_bin['prf_cm_bins_ci_upper_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['pcm_median'].values, x['prf_loo_rsq'].values, 75)).values
    df_polar_cm_bin['prf_cm_bins_ci_lower_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['pcm_median'].values, x['prf_loo_rsq'].values, 25)).values

    if num_roi == 0: df_polar_cm_bins = df_polar_cm_bin
    else: df_polar_cm_bins = pd.concat([df_polar_cm_bins, df_polar_cm_bin]) 

df_pcm_polar = df_polar_cm_bins



In [ ]:
fig = prf_polar_pcm_plot(df=df_pcm_polar, figure_info=analysis_info, rsq2use='prf_loo_rsq')
fig.show()

# polar SFp

In [ ]:
def prf_polar_SFp_plot(df, figure_info, rsq2use):
    """
    Make scatter plot for linear relationship between eccentricity and size

    Parameters
    ----------
    df : A data dataframe
    figure_info : dict with figure settings
    rsq2use : rsquare to use
    
    Returns
    -------
    fig : eccentricy as a function of size plot
    """
    
    from maths_utils import weighted_regression

    max_ecc = 10 #############
    # General figure settings
    template_specs = dict(axes_color="rgba(0, 0, 0, 1)",
                          axes_width=2,
                          axes_font_size=15,
                          bg_col="rgba(255, 255, 255, 1)",
                          font='Arial',
                          title_font_size=15,
                          rois_plot_width=1.5)
    
    # General figure settings
    fig_template = plotly_template(template_specs)
    max_size = figure_info['ecc_size_max'][1]
    roi_colors = figure_info['roi_colors']
    fig_margin = figure_info['rois_fig_margin']
    rois_groups = figure_info['rois_groups_plot']
    rows, cols = 1, len(rois_groups)
    rois_hor_spacing = figure_info['rois_hor_spacing']
    rois_ver_spacing = figure_info['rois_ver_spacing']
    rois_plot_height = figure_info['rois_plot_height']
    rois_plot_width = figure_info['rois_plot_width']    
    ecc_axis = [0, figure_info['ecc_size_max'][0]]
    size_axis = [0, analysis_info['flatmap_ncsf_auc_scale'][1]]
    
    fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows-1))
    fig_width = rois_plot_width * cols + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols-1))
    hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
    ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])

    # General settings
    
    fig = make_subplots(rows=rows, cols=cols, print_grid=False, 
                       horizontal_spacing=hor_spacing,
                       vertical_spacing=ver_spacing)
    
    for l, line_label in enumerate(rois_groups):
        for j, roi in enumerate(line_label):
            
            # Parametring colors
            roi_color = roi_colors[roi]
            roi_color_opac = f"rgba{roi_color[3:-1]}, 0.15)"
            
            # Get data
            df_roi = df.loc[(df.roi == roi)]
            polar_median = np.array(df_roi.prf_polar_bins)
            SFp_median = np.array(df_roi.SFp_bins_median)
            r2_median = np.array(df_roi[f'{rsq2use}_bins_median'])
            SFp_upper_bound = np.array(df_roi.SFp_bins_ci_upper_bound)
            SFp_lower_bound = np.array(df_roi.SFp_bins_ci_lower_bound)
    
            # Markers
            fig.add_trace(go.Scatter(x=polar_median, 
                                     y=SFp_median, 
                                     mode='markers+lines', 
                                     error_y=dict(type='data', 
                                                  array=SFp_upper_bound - SFp_median, 
                                                  arrayminus=SFp_median - SFp_lower_bound,
                                                  visible=True, 
                                                  thickness=3, 
                                                  width=0, 
                                                  color=roi_color),
                                      marker=dict(color=roi_color,
                                                  symbol='square',
                                                  size=8, 
                                                  line=dict(color=roi_color, 
                                                            width=3)), 
                                      showlegend=False), 
                          row=1, col=l + 1)
            
            # Add legend
            annotation = go.layout.Annotation(x=1, y=max_ecc-j*1.5, text=roi, xanchor='left',
                                              showarrow=False, font_color=roi_color, 
                                              font_family=template_specs['font'],
                                              font_size=template_specs['axes_font_size'],
                                             )
            fig.add_annotation(annotation, row=1, col=l+1)

        # Set axis titles only for the left-most column and bottom-most row
        fig.update_yaxes(title_text='pRF cortical magn. (mm/dva)', row=1, col=1)
        fig.update_xaxes(title_text='pRF polar angle (dva)', range=[0, 360], tickvals=[0, 90, 180, 270, 360], ticktext=['0', '90', '180', '270', '360'], showline=True, row=1, col=l+1)
        fig.update_yaxes(range=[0, 10], showline=True)
        fig.update_layout(height=fig_height, 
                          width=fig_width, 
                          showlegend=False, 
                          template=fig_template,
                          margin_l=fig_margin[0], 
                          margin_t=fig_margin[1], 
                          margin_r=fig_margin[2], 
                          margin_b=fig_margin[3]
                         )
        
    return fig

In [ ]:
data = ncsf_tsv_df
num_polar_SFp_bins = 16
for num_roi, roi in enumerate(rois):
    df_roi = data.loc[(data.roi == roi)]
    df_bins = df_roi.groupby(pd.cut(df_roi['prf_polar_angle'], bins=num_polar_SFp_bins))
    df_polar_SFp_bin = pd.DataFrame()
    df_polar_SFp_bin['roi'] = [roi]*num_polar_SFp_bins
    df_polar_SFp_bin['num_bins'] = np.arange(num_polar_SFp_bins)
    df_polar_SFp_bin['prf_polar_bins'] = df_bins.apply(lambda x: weighted_nan_median(x['prf_polar_angle'].values, x['prf_loo_rsq'].values)).values
    df_polar_SFp_bin['SFp_bins_median'] = df_bins.apply(lambda x: weighted_nan_median(x['SFp'].values, x['prf_loo_rsq'].values)).values
    df_polar_SFp_bin['prf_loo_rsq_bins_median'] = np.array(df_bins['prf_loo_rsq'].median())
    df_polar_SFp_bin['SFp_bins_ci_upper_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['SFp'].values, x['prf_loo_rsq'].values, 75)).values
    df_polar_SFp_bin['SFp_bins_ci_lower_bound'] = df_bins.apply(lambda x: weighted_nan_percentile(x['SFp'].values, x['prf_loo_rsq'].values, 25)).values

    if num_roi == 0: df_polar_SFp_bins = df_polar_SFp_bin
    else: df_polar_SFp_bins = pd.concat([df_polar_SFp_bins, df_polar_SFp_bin]) 

df_SFp_polar = df_polar_SFp_bins



In [ ]:
fig = prf_polar_SFp_plot(df=df_SFp_polar, figure_info=analysis_info, rsq2use='prf_loo_rsq')
fig.show()

# Parameter Median 

In [ ]:
# Parameters median 'width_r', 'SFp', 'CSp', 'width_l', 'crf_exp',
# ------------------
for num_roi, roi in enumerate(rois):
    df_roi = ncsf_tsv_df.loc[(ncsf_tsv_df.roi == roi)]

    df_params_median_roi = pd.DataFrame()
    df_params_median_roi['roi'] = [roi]

    df_params_median_roi[f'{rsq2use}_weighted_median'] = weighted_nan_median(df_roi[rsq2use], weights=df_roi[rsq2use])
    df_params_median_roi[f'{rsq2use}_ci_down'] = weighted_nan_percentile(df_roi[rsq2use], df_roi[rsq2use], 25)
    df_params_median_roi[f'{rsq2use}_ci_up'] = weighted_nan_percentile(df_roi[rsq2use], df_roi[rsq2use], 75)
    
    df_params_median_roi['width_r_weighted_median'] = weighted_nan_median(df_roi.width_r, weights=df_roi[rsq2use])
    df_params_median_roi['width_r_ci_down'] = weighted_nan_percentile(df_roi.width_r, df_roi[rsq2use], 25)
    df_params_median_roi['width_r_ci_up'] = weighted_nan_percentile(df_roi.width_r, df_roi[rsq2use], 75)

    df_params_median_roi['SFp_weighted_median'] = weighted_nan_median(df_roi.SFp, weights=df_roi[rsq2use])
    df_params_median_roi['SFp_ci_down'] = weighted_nan_percentile(df_roi.SFp, df_roi[rsq2use], 25)
    df_params_median_roi['SFp_ci_up'] = weighted_nan_percentile(df_roi.SFp, df_roi[rsq2use], 75)
    
    df_params_median_roi['CSp_median'] = weighted_nan_median(df_roi.CSp, weights=df_roi[rsq2use])
    df_params_median_roi['CSp_ci_down'] = weighted_nan_percentile(df_roi.CSp, df_roi[rsq2use], 25)
    df_params_median_roi['CSp_ci_up'] = weighted_nan_percentile(df_roi.CSp, df_roi[rsq2use], 75)
     
    df_params_median_roi['width_l_weighted_median'] = weighted_nan_median(df_roi.width_l, weights=df_roi[rsq2use])
    df_params_median_roi['width_l_ci_down'] = weighted_nan_percentile(df_roi.width_l, df_roi[rsq2use], 25)
    df_params_median_roi['width_l_ci_up'] = weighted_nan_percentile(df_roi.width_l, df_roi[rsq2use], 75)

    df_params_median_roi['crf_exp_weighted_median'] = weighted_nan_median(df_roi.crf_exp, weights=df_roi[rsq2use])
    df_params_median_roi['crf_exp_ci_down'] = weighted_nan_percentile(df_roi.crf_exp, df_roi[rsq2use], 25)
    df_params_median_roi['crf_exp_ci_up'] = weighted_nan_percentile(df_roi.crf_exp, df_roi[rsq2use], 75)

    df_params_median_roi['auc_weighted_median'] = weighted_nan_median(df_roi.auc, weights=df_roi[rsq2use])
    df_params_median_roi['auc_ci_down'] = weighted_nan_percentile(df_roi.auc, df_roi[rsq2use], 25)
    df_params_median_roi['auc_ci_up'] = weighted_nan_percentile(df_roi.auc, df_roi[rsq2use], 75)

    df_params_median_roi['normalize_auc_weighted_median'] = weighted_nan_median(df_roi.normalize_auc, weights=df_roi[rsq2use])
    df_params_median_roi['normalize_auc_ci_down'] = weighted_nan_percentile(df_roi.normalize_auc, df_roi[rsq2use], 25)
    df_params_median_roi['normalize_auc_ci_up'] = weighted_nan_percentile(df_roi.normalize_auc, df_roi[rsq2use], 75)

    if num_roi == 0: df_params_median = df_params_median_roi
    else: df_params_median = pd.concat([df_params_median, df_params_median_roi])



In [ ]:
def ncsf_params_median_plot(df, figure_info, rsq2use):

    template_specs = dict(
        axes_color="rgba(0, 0, 0, 1)",
        axes_width=2,
        axes_font_size=15,
        bg_col="rgba(255, 255, 255, 1)",
        font='Arial',
        title_font_size=15,
        rois_plot_width=1.5
    )

    fig_template = plotly_template(template_specs)

    rows, cols = 3, 3
    rois = figure_info['rois']
    roi_colors = figure_info['roi_colors']

    fig_margin = figure_info['rois_fig_margin']
    hspace = figure_info['rois_hor_spacing']
    vspace = figure_info['rois_ver_spacing']
    bar_width = figure_info['rois_bar_width']
    h = figure_info['rois_plot_height']

    fig_height = h * rows + fig_margin[1] + fig_margin[3] + vspace * (rows - 1)
    fig_width  = bar_width * cols * len(rois) + fig_margin[0] + fig_margin[2] + hspace * (cols - 1)

    fig = make_subplots(
        rows=rows, cols=cols,
        vertical_spacing=vspace / (fig_height - fig_margin[1] - fig_margin[3]),
        horizontal_spacing=hspace / (fig_width - fig_margin[0] - fig_margin[2])
    )

    def add_metric(row, col, roi, y_med, y_lo, y_hi, name, showlegend=False):

        fig.add_trace(go.Scatter(
            x=[roi],
            y=tuple(y_med),
            mode='markers',
            error_y=dict(
                type='data',
                array=[y_hi - y_med],
                arrayminus=[y_med - y_lo],
                visible=True,
                thickness=3,
                width=0,
                color=roi_colors[roi]
            ),
            marker=dict(
                symbol="square",
                color=roi_colors[roi],
                size=12,
                line=dict(color=roi_colors[roi], width=3)
            ),
            name=roi,
            legendgroup=name,
            showlegend=showlegend
        ), row=row, col=col)

    for roi in rois:

        df_roi = df.loc[df.roi == roi]

        # 1) rsq2use
        add_metric(
            1, 1, roi,
            df_roi[f'{rsq2use}_weighted_median'],
            df_roi[f'{rsq2use}_ci_down'],
            df_roi[f'{rsq2use}_ci_up'],
            "rsq"
        )

        # 2) width_r
        add_metric(
            1, 2, roi,
            df_roi['width_r_weighted_median'],
            df_roi['width_r_ci_down'],
            df_roi['width_r_ci_up'],
            "width_r"
        )

        # 3) SFp
        add_metric(
            1, 3, roi,
            df_roi['SFp_weighted_median'],
            df_roi['SFp_ci_down'],
            df_roi['SFp_ci_up'],
            "SFp"
        )

        # 4) CSp (median not weighted in your pipeline)
        add_metric(
            2, 1, roi,
            df_roi['CSp_median'],
            df_roi['CSp_ci_down'],
            df_roi['CSp_ci_up'],
            "CSp"
        )

        # 5) width_l
        add_metric(
            2, 2, roi,
            df_roi['width_l_weighted_median'],
            df_roi['width_l_ci_down'],
            df_roi['width_l_ci_up'],
            "width_l"
        )

        # 6) crf_exp
        add_metric(
            2, 3, roi,
            df_roi['crf_exp_weighted_median'],
            df_roi['crf_exp_ci_down'],
            df_roi['crf_exp_ci_up'],
            "crf_exp"
        )

        # 6) auc
        add_metric(
            3, 1, roi,
            df_roi['auc_weighted_median'],
            df_roi['auc_ci_down'],
            df_roi['auc_ci_up'],
            "auc"
        )

        # 6) normalize_auc
        add_metric(
            3, 2, roi,
            df_roi['normalize_auc_weighted_median'],
            df_roi['normalize_auc_ci_down'],
            df_roi['normalize_auc_ci_up'],
            "normalize_auc"
        )

    titles = [
        rsq2use,
        "width_r",
        "SFp",
        "CSp",
        "width_l",
        "crf_exp",
        "auc",
        "normalize_auc"
    ]

    for i, t in enumerate(titles):
        fig.update_yaxes(
            showline=True,
            title_text=t,
            row=i // 3 + 1,
            col=i % 3 + 1
        )

    fig.update_xaxes(showline=True, ticklen=0, linecolor='rgba(255,255,255,0)')

    fig.update_layout(
        height=fig_height,
        width=fig_width,
        template=fig_template,
        margin_l=fig_margin[0],
        margin_t=fig_margin[1],
        margin_r=fig_margin[2],
        margin_b=fig_margin[3],
        legend=dict(orientation="h", y=1.1)
    )

    return fig

In [ ]:
fig = ncsf_params_median_plot(df=df_params_median, figure_info=analysis_info, rsq2use=rsq2use)
fig.show()

In [ ]:
fig = ncsf_params_median_plot(df=df_params_median, figure_info=analysis_info, rsq2use=rsq2use)
fig.show()

# Time series

In [ ]:
fit_fn = '/Users/uriel/disks/meso_shared/nCSF/derivatives/pp_data/sub-01/fsnative/ncsf/fit/sub-01_task-nCSF_hemi-L_fmriprep_dct_z-score_avg_ncsf_fit.func.gii'
pred_fn = '/Users/uriel/disks/meso_shared/nCSF/derivatives/pp_data/sub-01/fsnative/ncsf/fit/sub-01_task-nCSF_hemi-L_fmriprep_dct_z-score_avg_ncsf_pred.func.gii'
bold_fn = '/Users/uriel/disks/meso_shared/nCSF/derivatives/pp_data/sub-01/fsnative/func/fmriprep_dct_z-score_avg/sub-01_task-nCSF_hemi-L_fmriprep_dct_z-score_avg_bold.func.gii'

In [ ]:
fit_img, fit_data = load_surface(fit_fn)
pred_img, pred_data = load_surface(pred_fn)
bold_img, bold_data = load_surface(bold_fn)

In [ ]:
fit_data[:,167993]

In [ ]:
roi_verts_dict = get_rois(subject, 
                          surf_format=format_, 
                          rois_type=rois_method_format,
                          mask=True, 
                          rois=rois, 
                          hemis='hemi-L')

In [ ]:
maps_names_ncsf = analysis_info['maps_names_ncsf']
for idx, col_name in enumerate(maps_names_ncsf):
    exec("{}_idx = idx".format(col_name))


In [ ]:
maps_names_ncsf

In [ ]:
roi = 'sIPS'
fit_data_roi = fit_data[:,roi_verts_dict[roi]]
pred_data_roi = pred_data[:,roi_verts_dict[roi]]
bold_data_roi = bold_data[:,roi_verts_dict[roi]]

In [ ]:
best_r2_idx = np.argmax(fit_data_roi[ncsf_rsq_idx, :])
print('best R2 is {} with idx {}'.format(fit_data_roi[ncsf_rsq_idx, best_r2_idx], best_r2_idx))

In [ ]:
for n_name, map_name in enumerate(maps_names_ncsf):
    print('{} is {}'.format(map_name, fit_data_roi[n_name, best_r2_idx]))


In [ ]:
plt.plot(np.arange(0,pred_data.shape[0]), bold_data[:,best_r2_idx])
plt.plot(np.arange(0,pred_data.shape[0]), pred_data[:,best_r2_idx])


In [ ]:
np.savetxt('/Users/uriel/Downloads/vertext_ts_{}.csv'.format(roi), np.stack([bold_data[:,best_r2_idx], pred_data[:,best_r2_idx]], axis=1), delimiter=',')

In [ ]:
plt.scatter(np.arange(0,pred_data.shape[0]), pred_data[:,best_r2_idx])

In [ ]:
fig = draw_timeseries2(bold_data=bold_data_roi, model_prediction=pred_data_roi, vox_data=best_r2_idx, vox_model=best_r2_idx, TRs=bold_data_roi.shape[0], TR=1.6, roi=roi)
fig.show()

In [ ]:
bold_data_roi.shape[0]/2

In [ ]:
template_specs = dict(axes_color="rgba(0, 0, 0, 1)",
                      axes_width=2,
                      axes_font_size=15,
                      bg_col="rgba(255, 255, 255, 1)",
                      font='Arial',
                      title_font_size=15,
                      rois_plot_width=1.5)
def draw_timeseries2(bold_data, model_prediction, vox_data, vox_model, TRs, TR, roi):
    
    # General figure settings
    fig_template = plotly_template(template_specs)
    
    # Subplot settings
    rows, cols = 2, 2
    margin_t, margin_b, margin_l, margin_r = 50, 50, 50, 50
    fig_ratio = 5
    fig_height = 1080/fig_ratio + (1080/fig_ratio*0.15) + margin_t + margin_b
    fig_width = 1920/fig_ratio + 1920/fig_ratio + margin_l + margin_r
    column_widths, row_heights = [1, 1], [0.15, 1]
    sb_specs = [[{}, {}], [{}, {}]]
    hover_data = 'Time: %{x:1.2f} s<br>' + 'z-score: %{y:1.2f}'
    hover_model = 'Time: %{x:1.2f} s<br>' + 'z-score: %{y:1.2f}'

    xaxis_range = [0, 350]
    yaxis_range = [-2, 3]
    yaxis_dtick = 1
    x_tickvals = np.linspace(0, TRs, 6) * TR

    model_xrange = [-10, 10]
    model_yrange = [-10, 10]    

    rolling = 3
    data_col = 'rgba(0, 0, 0, 1)'
    model_col = 'rgba(200, 0, 0, 1)'
    subplot_titles = ['<b>{} time series </b> ({})'.format(roi, subject), '', '', '']

    # avg to have less points
    bold_data_reshaped = bold_data.reshape(int(TRs/2), 2, -1)
    bold_data_mean = np.mean(bold_data_reshaped, axis=1)

    model_pred_data_reshaped = model_prediction.reshape(int(TRs/2), 2, -1)
    model_pred_data_mean = np.mean(model_pred_data_reshaped, axis=1)

    # create figure
    fig = make_subplots(
        rows=rows, cols=cols, specs=sb_specs, print_grid=False,
        vertical_spacing=0.05, horizontal_spacing=0.05,
        column_widths=column_widths, row_heights=row_heights,
        subplot_titles=subplot_titles
    )

    # time series data
    fig.append_trace(go.Scatter(
        x=np.linspace(0, TRs*TR, 104),
        y=bold_data_mean[:, vox_data],
        name='<i>data<i>',
        showlegend=False, mode='markers', marker_color=data_col,
        hovertemplate=hover_data,
        cliponaxis=False,
        line_width=0, opacity=1, marker_size=6
    ), row=2, col=1)

    # time series predictions
    fig.append_trace(go.Scatter(
        x=np.linspace(0, TRs*TR, 104),
        y=model_pred_data_mean[:, vox_model],
        name='<i>model<i>',
        showlegend=False, mode='lines', line_color=data_col,
        hovertemplate=hover_model,
        cliponaxis=False,
        line_width=2, opacity=1
    ), row=2, col=1)

    # set axis
    for row in np.arange(rows):
        for col in np.arange(cols):
            fig.update_xaxes(
                visible=True, ticklen=8, linewidth=template_specs['axes_width'],
                row=row+1, col=col+1
            )
            fig.update_yaxes(
                visible=True, ticklen=8, linewidth=template_specs['axes_width'],
                row=row+1, col=col+1
            )
            
    fig.update_xaxes(scaleanchor="y4", scaleratio=1, row=2, col=2)
    fig.update_yaxes(scaleanchor="x4", scaleratio=1, row=2, col=2)
    fig.layout.update(
        xaxis_range=xaxis_range, xaxis_title='', 
        xaxis_visible=False, yaxis_visible=False,
        yaxis_range=[0, 1], yaxis_title='',
        xaxis4_range=model_xrange, xaxis4_title='', 
        yaxis4_range=model_yrange, yaxis4_title='', 
        xaxis4_visible=False, yaxis4_visible=False,
        xaxis3_tickvals=x_tickvals, xaxis3_ticktext=np.round(x_tickvals),
        xaxis3_range=xaxis_range, xaxis3_title='Time (seconds)',
        yaxis3_range=yaxis_range, yaxis3_title='z-score', yaxis3_dtick=yaxis_dtick,
        template=fig_template, width=fig_width, height=fig_height,
        margin_l=margin_l+10, margin_r=margin_r-10, margin_t=margin_t-20, margin_b=margin_b+20,
        legend_yanchor='top', legend_y=0.85, legend_xanchor='left',
        legend_x=0.02, legend_bgcolor='rgba(255,255,255,0)'
    )

    return fig

# Violins 

In [ ]:
df_violins = ncsf_tsv_df 
analysis_info['flatmap_ncsf_width_l_scale'] = [0.5, 1]
analysis_info['flatmap_ncsf_width_r_scale'] = [0, 1.5]

In [ ]:
def prf_violins_plot(df, figure_info, rsq2use):
    """
    Make violins plots for pRF loo_r2, size, n and pcm

    Parameters
    ----------
    df : dataframe
    figure_info : dict with figure settings
    rsq2use : rsquare value to use
    
    Returns
    -------
    fig : violins plot
    """
    
    # General figure settings
    template_specs = dict(axes_color="rgba(0, 0, 0, 1)",
                          axes_width=2,
                          axes_font_size=15,
                          bg_col="rgba(255, 255, 255, 1)",
                          font='Arial',
                          title_font_size=15,
                          rois_plot_width=1.5)
    
    # General figure settings
    fig_template = plotly_template(template_specs)
    rows = 2
    cols = 3
    rois = figure_info['rois']
    roi_colors = figure_info['roi_colors']
    fig_margin = figure_info['rois_fig_margin']
    rois_hor_spacing = figure_info['rois_hor_spacing']
    rois_ver_spacing = figure_info['rois_ver_spacing']
    bar_width = figure_info['rois_bar_width']
    rois_plot_height = figure_info['rois_plot_height']
    
    fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows-1))
    fig_width = bar_width * cols * len(rois) + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols-1))
    hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
    ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])
    
    fig = make_subplots(rows=rows, 
                        cols=cols, 
                        print_grid=False, 
                        vertical_spacing=ver_spacing,
                        horizontal_spacing=hor_spacing)

    for j, roi in enumerate(rois):
        
        df_roi = df.loc[(df.roi == roi)]
        
        # nCSF r2 or loor2    
        fig.add_trace(go.Violin(x=df_roi.roi[df_roi.roi==roi], 
                                y=df_roi[rsq2use], 
                                name=roi, 
                                opacity=1,
                                showlegend=False, 
                                points=False, 
                                spanmode='manual', 
                                span=figure_info['violin_rsq_range'],
                                scalemode='width', 
                                fillcolor=roi_colors[roi],
                                line_color=roi_colors[roi]), 
                      row=1, col=1)
                
        # width_r
        fig.add_trace(go.Violin(x=df_roi.roi[df_roi.roi==roi], 
                                y=df_roi.width_r, 
                                name=roi, 
                                opacity=1,
                                showlegend=False, 
                                points=False, 
                                spanmode='manual', 
                                span=figure_info['flatmap_ncsf_width_r_scale'],
                                scalemode='width', 
                                fillcolor=roi_colors[roi],
                                line_color=roi_colors[roi]), 
                      row=1, col=2)
        
        # SFp
        fig.add_trace(go.Violin(x=df_roi.roi[df_roi.roi==roi], 
                                y=df_roi.SFp, 
                                name=roi, 
                                opacity=1,
                                showlegend=False, 
                                points=False, 
                                spanmode='manual', 
                                span=figure_info['flatmap_ncsf_SFp_scale'],
                                scalemode='width', 
                                fillcolor=roi_colors[roi],
                                line_color=roi_colors[roi]), 
                      row=1, col=3)
        
        # CSp
        fig.add_trace(go.Violin(x=df_roi.roi[df_roi.roi==roi], 
                                y=df_roi.CSp, 
                                name=roi, 
                                opacity=1,
                                showlegend=False, 
                                points=False, 
                                spanmode='manual', 
                                span=figure_info['flatmap_ncsf_CSp_scale'],
                                scalemode='width', 
                                fillcolor=roi_colors[roi],
                                line_color=roi_colors[roi]), 
                      row=2, col=1)

        # width_l
        fig.add_trace(go.Violin(x=df_roi.roi[df_roi.roi==roi], 
                                y=df_roi.width_l, 
                                name=roi, 
                                opacity=1,
                                showlegend=False, 
                                points=False, 
                                spanmode='manual', 
                                span=figure_info['flatmap_ncsf_width_l_scale'],
                                scalemode='width', 
                                fillcolor=roi_colors[roi],
                                line_color=roi_colors[roi]), 
                      row=2, col=2)

        # crf_exp
        fig.add_trace(go.Violin(x=df_roi.roi[df_roi.roi==roi], 
                        y=df_roi.crf_exp, 
                        name=roi, 
                        opacity=1,
                        showlegend=False, 
                        points=False, 
                        spanmode='manual', 
                        span=[0,4],#figure_info['flatmap_ncsf_CRFsplope_scale'],
                        scalemode='width', 
                        fillcolor=roi_colors[roi],
                        line_color=roi_colors[roi]), 
              row=2, col=3)
        # Set axis titles only for the left-most column and bottom-most row
        if 'loo' in rsq2use:
            title_y = 'nCSF LOO R<sup>2</sup>'
        else:
            title_y = 'pRF R<sup>2</sup>'

        'width_r', 'SFp', 'CSp', 'width_l', 'crf_exp'
        fig.update_yaxes(showline=True, 
                         range=figure_info['flatmap_ncsf_rsq_scale'], #[0,0.2],
                         nticks=10, 
                         title_text=title_y,
                         row=1, col=1)
        
        fig.update_yaxes(showline=True, 
                         range=figure_info['flatmap_ncsf_width_r_scale'],
                         nticks=7, 
                         title_text='nCSF width_r', 
                         row=1, col=2)
        
        fig.update_yaxes(showline=True, 
                         range=figure_info['flatmap_ncsf_SFp_scale'],
                         nticks=7, 
                         title_text='nCSF SFp', 
                         row=1, col=3)
        
        fig.update_yaxes(showline=True, 
                         range=figure_info['flatmap_ncsf_CSp_scale'],
                         nticks=5, 
                         title_text='nCSF CSp', 
                         row=2, col=1)

        fig.update_yaxes(showline=True, 
                 range=figure_info['flatmap_ncsf_width_l_scale'],
                 nticks=5, 
                 title_text='nCSF width_l', 
                 row=2, col=2)
        
        fig.update_yaxes(showline=True, 
                         range=[0,4],#figure_info['flatmap_ncsf_CRFsplope_scale'],
                         nticks=5, 
                         title_text='nCSF crf_exp', 
                         row=2, col=3)
        
        
        fig.update_xaxes(showline=True, 
                         ticklen=0, 
                         linecolor=('rgba(255,255,255,0)'))
        
    fig.update_layout(height=fig_height, 
                      width=fig_width, 
                      showlegend=False,
                      legend=dict(orientation="h", 
                                  font_family=template_specs['font'],
                                  font_size=template_specs['axes_font_size'],
                                  y=1.1, 
                                  yanchor='top', 
                                  xanchor='left', 
                                  traceorder='normal', 
                                  itemwidth=30), 
                      template=fig_template,
                      margin_l=fig_margin[0], 
                      margin_t=fig_margin[1], 
                      margin_r=fig_margin[2], 
                      margin_b=fig_margin[3]
                     )

    return fig

In [ ]:
fig = prf_violins_plot(df=df_violins, figure_info=analysis_info, rsq2use=rsq2use)
fig.show()

# Histograms

In [ ]:
def histogram_plot(df, figure_info, rsq2use):

    template_specs = dict(
        axes_color="rgba(0, 0, 0, 1)",
        axes_width=2,
        axes_font_size=15,
        bg_col="rgba(255, 255, 255, 1)",
        font='Arial',
        title_font_size=15,
        rois_plot_width=1.5
    )

    fig_template = plotly_template(template_specs)

    rows = 3
    cols = 3

    roi_colors = figure_info['roi_colors']
    fig_margin = figure_info['rois_fig_margin']
    rois_hor_spacing = figure_info['rois_hor_spacing']
    rois_ver_spacing = figure_info['rois_ver_spacing']
    bar_width = figure_info['rois_bar_width']
    rois_plot_height = figure_info['rois_plot_height']

    rois_groups = figure_info['rois_groups_plot']  # <<< IMPORTANT

    figs = []  # on stocke toutes les figures

    for rois in rois_groups:

        fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows - 1))
        fig_width = bar_width * cols * len(rois) + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols - 1))

        hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
        ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])

        fig = make_subplots(
            rows=rows,
            cols=cols,
            vertical_spacing=ver_spacing,
            horizontal_spacing=hor_spacing
        )

        for roi in rois:

            df_roi = df.loc[df.roi == roi]

            def add_hist(x, row, col):
                fig.add_trace(
                    go.Histogram(
                        x=x,
                        opacity=0.75,
                        showlegend=False,
                        marker_color=roi_colors[roi],
                        nbinsx=40,
                        histnorm='probability density'
                    ),
                    row=row,
                    col=col
                )

            add_hist(df_roi[rsq2use], 1, 1)
            add_hist(df_roi.width_r,   1, 2)
            add_hist(df_roi.SFp,       1, 3)

            add_hist(df_roi.CSp,       2, 1)
            add_hist(df_roi.width_l,   2, 2)
            add_hist(df_roi.crf_exp,   2, 3)

            add_hist(df_roi.auc,       3, 1)
            add_hist(df_roi.normalize_auc,   3, 2)
            
        title_y = 'nCSF LOO R²' if 'loo' in rsq2use else 'pRF R²'

        fig.update_layout(
            height=fig_height,
            width=fig_width,
            barmode='overlay',
            template=fig_template,
            margin_l=fig_margin[0],
            margin_t=fig_margin[1],
            margin_r=fig_margin[2],
            margin_b=fig_margin[3]
        )

        # axes (identique logique que toi)
        fig.update_xaxes(title_text=title_y, showline=True, row=1, col=1)
        fig.update_xaxes(title_text='width_r', showline=True, row=1, col=2)
        fig.update_xaxes(title_text='SFp', showline=True, row=1, col=3)
        fig.update_xaxes(title_text='CSp', showline=True, row=2, col=1)
        fig.update_xaxes(title_text='width_l', showline=True, row=2, col=2)
        fig.update_xaxes(title_text='crf_exp', showline=True, range=[0, 4], row=2, col=3)
        fig.update_xaxes(title_text='auc', showline=True, range=[0, 8], row=3, col=1)
        fig.update_xaxes(title_text='normalize_auc', showline=True, range=[0, 160], row=3, col=2)

        for r in range(1, rows + 1):
            for c in range(1, cols + 1):
                fig.update_yaxes(title_text='Density', showline=True, row=r, col=c)

        figs.append(fig)

    return figs

In [ ]:
figs = histogram_plot(df=df_violins, figure_info=analysis_info, rsq2use=rsq2use)


for fig in figs : 
    fig.show()

In [ ]:
def histogram_plot_prf(df, figure_info, rsq2use):

    template_specs = dict(
        axes_color="rgba(0, 0, 0, 1)",
        axes_width=2,
        axes_font_size=15,
        bg_col="rgba(255, 255, 255, 1)",
        font='Arial',
        title_font_size=15,
        rois_plot_width=1.5
    )

    fig_template = plotly_template(template_specs)

    rows = 2
    cols = 3

    roi_colors = figure_info['roi_colors']
    fig_margin = figure_info['rois_fig_margin']
    rois_hor_spacing = figure_info['rois_hor_spacing']
    rois_ver_spacing = figure_info['rois_ver_spacing']
    bar_width = figure_info['rois_bar_width']
    rois_plot_height = figure_info['rois_plot_height']

    rois_groups = figure_info['rois_groups_plot']  # <<< IMPORTANT

    figs = []  # on stocke toutes les figures

    for rois in rois_groups:

        fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows - 1))
        fig_width = bar_width * cols * len(rois) + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols - 1))

        hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
        ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])

        fig = make_subplots(
            rows=rows,
            cols=cols,
            vertical_spacing=ver_spacing,
            horizontal_spacing=hor_spacing
        )

        for roi in rois:

            df_roi = df.loc[df.roi == roi]

            def add_hist(x, row, col):
                fig.add_trace(
                    go.Histogram(
                        x=x,
                        opacity=0.75,
                        showlegend=False,
                        marker_color=roi_colors[roi],
                        nbinsx=40,
                        histnorm='probability density'
                    ),
                    row=row,
                    col=col
                )

                

            add_hist(df_roi.prf_loo_rsq, 1, 1)
            add_hist(df_roi.prf_ecc,   1, 2)
            add_hist(df_roi.prf_size,       1, 3)

            add_hist(df_roi.prf_n,       2, 1)
            add_hist(df_roi.pcm_median,   2, 2)
            # add_hist(df_roi.crf_exp,   2, 3)

        fig.update_layout(
            height=fig_height,
            width=fig_width,
            barmode='overlay',
            template=fig_template,
            margin_l=fig_margin[0],
            margin_t=fig_margin[1],
            margin_r=fig_margin[2],
            margin_b=fig_margin[3]
        )

        # axes (identique logique que toi)
        fig.update_xaxes(title_text='prf_loo_rsq', showline=True, row=1, col=1)
        fig.update_xaxes(title_text='prf_ecc', showline=True, row=1, col=2)
        fig.update_xaxes(title_text='prf_size', showline=True, row=1, col=3)
        fig.update_xaxes(title_text='prf_n', showline=True, row=2, col=1)
        fig.update_xaxes(title_text='pcm_median', showline=True, row=2, col=2)
        # fig.update_xaxes(title_text='crf_exp', showline=True, range=[0, 4], row=2, col=3)

        for r in range(1, rows + 1):
            for c in range(1, cols + 1):
                fig.update_yaxes(title_text='Density', showline=True, row=r, col=c)

        figs.append(fig)

    return figs

In [ ]:
figs = histogram_plot_prf(df=df_violins, figure_info=analysis_info, rsq2use=rsq2use)


for fig in figs : 
    fig.show()

# polar plot

In [ ]:
ncsf_tsv_df['prf_polar_angle'] = np.mod(np.degrees(np.angle(ncsf_tsv_df.polar_real + 1j * ncsf_tsv_df.polar_imag)), 360) 

In [ ]:
roi = 'mPCS'
df_roi = ncsf_tsv_df.loc[(ncsf_tsv_df.roi == roi)]# & (df_roi[rsq2use] > 0.01)]

fig = go.Figure()

fig.add_trace(
    go.Scatterpolar(
        theta=df_roi['prf_polar_angle'].to_list(),
        r=df_roi['prf_ecc'].to_list(),
        mode="markers",
        marker=dict(
            size=(df_roi[rsq2use] * 30).to_list(),
            color=df_roi["SFp"].to_list(),
            colorscale="viridis", 
            cmin=0, 
            cmax=5,
            showscale=True,
            colorbar=dict(title="Peak SF (cpd)")
        ),
        opacity=1,
        name=roi,
        showlegend=True
    ))
fig.show()


In [ ]:
b = df_roi['prf_ecc'].to_list()

In [ ]:
df_roi['prf_ecc'].isna().sum()

In [ ]:
a = np.array([250, 45])
b = np.array([1, 10])